In [1]:
%env AWS_PROFILE=platform-developer

env: AWS_PROFILE=platform-developer


In [2]:
import subprocess
import sys
from pathlib import Path
from typing import Any
import lxml.etree as ET
from utils.marc import parse_single_marc_record

STYLESHEET_PATH = "/Users/brychtas/Documents/GitHub/axiell-collections-xslt/data/stylesheets/axc_to_marcxml_collect.xsl"


def prepare_for_xslt(record_tree: Any) -> ET._Element:
    """Wrap a raw Axiell record tree in the adlibXML/recordList structure the
    XSLT expects, stripping the OAI-PMH default namespace from all elements."""
    root = record_tree.getroot()
    # Strip the OAI default namespace so XSLT template names match
    for elem in root.iter():
        elem.tag = ET.QName(elem).localname
    adlib = ET.Element("adlibXML")
    record_list = ET.SubElement(adlib, "recordList")
    record_list.append(root)
    return adlib


def apply_xslt(
        xml_doc: Any,
        **xslt_params: Any,
):
    parser = ET.XMLParser(remove_blank_text=True)
    xslt_doc = ET.parse(STYLESHEET_PATH, parser)
    transform = ET.XSLT(xslt_doc)

    formatted_params = {
        key: ET.XSLT.strparam(str(value)) for key, value in xslt_params.items()
    }

    return transform(xml_doc, **formatted_params)


In [3]:
import os
from pathlib import Path
from lxml import etree
import json


def load_all_calm_works():
    calm_works_folder = "/Users/brychtas/Documents/all-raw-calm-works"
    calm_file_names = [f.name for f in Path(calm_works_folder).iterdir() if f.is_file()]

    calm_works = {}
    for file_name in calm_file_names:
        with open(os.path.join(calm_works_folder, file_name)) as f:
            parsed_work = json.load(f)
            calm_works[parsed_work["id"]] = parsed_work

    return calm_works


def load_all_axiell_works():
    axiell_works_folder = "/Users/brychtas/Documents/all-raw-axiell-works"
    axiell_file_names = [f.name for f in Path(axiell_works_folder).iterdir() if f.is_file()]
    axiell_works = {}

    for file_name in axiell_file_names:
        if not file_name.endswith(".xml"):
            continue

        parsed_work = etree.parse(os.path.join(axiell_works_folder, file_name))
        identifier = file_name.split(".")[0].replace("_", ":")
        axiell_works[identifier] = parsed_work

    return axiell_works

In [4]:
calm_works = load_all_calm_works()
print(len(calm_works))

axiell_works = load_all_axiell_works()
print(len(axiell_works))

408029
227789


In [5]:
from collections import Counter, defaultdict


def find_xml_value(record, field_path: list[str]):
    ns = {"oai": "http://www.openarchives.org/OAI/2.0/"}

    path = "/".join([f"oai:{val}" for val in field_path])
    element = record.find(path, namespaces=ns)
    if element is not None:
        return element.text


def count_oai_values(field_path: list[str]):
    values = []
    value_identifier_mapping = defaultdict(list)

    for identifier, record in axiell_works.items():
        value = find_xml_value(record, field_path)
        values.append(value)
        value_identifier_mapping[value].append(identifier)

    return Counter(values), value_identifier_mapping


def map_axiell_ids_to_calm_ids():
    mapping = {}
    for axiell_id, record in axiell_works.items():
        calm_id = find_xml_value(record, ["PIDother", "PID_other.non-URI_ID"])
        mapping[axiell_id] = calm_id

    return mapping

In [21]:
axiell_to_calm = map_axiell_ids_to_calm_ids()

In [ ]:
counter, mapping = count_oai_values(["accession_status", "value"])
print(counter, len(counter))
for i, axiell_id in enumerate(mapping["PRIVATE"]):
    print(axiell_id, axiell_to_calm[axiell_id],
          calm_works.get(axiell_to_calm[axiell_id], {}).get("data", {}).get("AccessStatus"))
    if i > 100:
        break

In [ ]:
counter, mapping = count_oai_values(["Record_progress", "record_progress", "value[@lang='0']"])
print(counter)
for i, axiell_id in enumerate(mapping["draft"]):
    print(axiell_to_calm[axiell_id],
          calm_works.get(axiell_to_calm[axiell_id], {}).get("data", {}).get("CatalogueStatus"))
    if i > 100:
        break

In [ ]:
counter, mapping = count_oai_values(["Inscription", "inscription.language"])
print(counter.keys())



In [ ]:
counter, mapping = count_oai_values(["record_type", "value[@lang='0']"])
print(counter)
for i, axiell_id in enumerate(mapping["Work"]):
    print(axiell_id, axiell_to_calm[axiell_id],
          calm_works.get(axiell_to_calm[axiell_id], {}).get("data", {}).get("Level"))
    if i > 10:
        break

In [ ]:
counter, mapping = count_oai_values(["Object_category", "object_category"])
print(counter)

In [ ]:
# All Mimsy works have consecutive identifiers < 20,000.
# No record with consecutive identifier < 20,000 has a predecessor identifier.

mimsy_count = 0
calm_count = 0
missing_pid_count = 0

for identifier, record in axiell_works.items():
    pid_value = find_xml_value(record, ["PIDother", "PID_other.non-URI_ID"])
    mimsy_value = find_xml_value(record, ["Description", "description.note"])

    numeric_id = int(identifier.split(":")[1])

    if mimsy_value:
        mimsy_count += 1
    if pid_value:
        calm_count += 1

    if numeric_id > 20000 and not (value := find_xml_value(record, ["current_location.context"])) and find_xml_value(
            record, ["current_location.name"]):
        print("!!!!!!!!!!")
        print(value)
        break
        #print(etree.tostring(record, pretty_print=True, encoding="unicode"))

    if not pid_value and numeric_id > 20000:
        #print(etree.tostring(record, pretty_print=True, encoding="unicode"))
        missing_pid_count += 1

print(mimsy_count)
print(calm_count)
print(missing_pid_count)

In [55]:
%pip install deepdiff

  Using cached orderly_set-5.5.0-py3-none-any.whl.metadata (6.6 kB)
Using cached orderly_set-5.5.0-py3-none-any.whl (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [deepdiff]
Note: you may need to restart the kernel to use updated packages.


In [6]:
SOURCE_WORKS_ROOT = "/Users/brychtas/Documents/all-source-works"

In [7]:
from utils.elasticsearch import get_client
from core.source import ElasticSource
import json
import os

es = get_client("read_only", pipeline_date="2025-10-02", es_mode="public")
source = ElasticSource(es, "works-source-2025-10-02", query={"match_all": {}})

for work in source.stream_raw():
    work_id = work['state']['sourceIdentifier']['value'].replace("/", "+")
    work_id_type = work['state']['sourceIdentifier']['identifierType']['id']

    os.makedirs(f"{SOURCE_WORKS_ROOT}/{work_id_type}", exist_ok=True)
    try:
        with open(f"{SOURCE_WORKS_ROOT}/{work_id_type}/{work_id}.json", "w") as f:
            f.write(json.dumps(work))
    except Exception as e:
        print(e)

In [32]:
from adapters.transformers.builders.axiell_work_builder import AxiellWorkBuilder
from pymarc import parse_xml_to_array
from io import BytesIO
from datetime import datetime

import logging
import structlog

structlog.configure(logger_factory=structlog.stdlib.LoggerFactory())
logging.basicConfig(level=logging.CRITICAL)

transformed_axiell_works = {}


def xslt_result_to_record(xslt_result):
    xml_bytes = bytes(xslt_result)
    return parse_xml_to_array(BytesIO(xml_bytes))[0]


i = 0

missing_id_field = []
missing_ref_no = []

for work_id, work_doc in axiell_works.items():
    # Skip Mimsy works
    if int(work_id.split(":")[1]) < 20000:
        continue

    xslt_result = apply_xslt(prepare_for_xslt(work_doc), control_003="UkLW")
    record = xslt_result_to_record(xslt_result)

    try:
        axiell_work = AxiellWorkBuilder(record=record, last_modified=datetime.now()).transform_work()
        transformed_axiell_works[work_id] = axiell_work
    except Exception as e:
        if "Missing RefNo" in str(e):
            missing_ref_no.append(work_id)
        elif "Empty id field" in str(e):
            missing_id_field.append(work_id)

    if i % 10000 == 0:
        print(i)
    i += 1

0
10000
20000
30000
40000
50000
60000
70000
80000
90000
100000
110000
120000
130000
140000
150000
160000
170000
180000
190000
200000
210000


In [39]:
print("Successfully transformed works:", len(transformed_axiell_works))
print("Axiell works with missing ID field:", len(missing_id_field))
print("Axiell works with a missing CALM RefNo:", len(missing_ref_no))

Successfully transformed works: 208924
Axiell works with missing ID field: 1955
Axiell works with a missing CALM RefNo: 3413


In [40]:
from deepdiff import DeepDiff


def standardise_works(calm_work, axiell_work):
    del calm_work["state"]["sourceIdentifier"]
    del axiell_work["state"]["sourceIdentifier"]
    del calm_work["state"]["relations"]["siblingsPreceding"]
    del calm_work["state"]["relations"]["siblingsSucceeding"]
    del calm_work["state"]["sourceModifiedTime"]
    del axiell_work["state"]["sourceModifiedTime"]
    del axiell_work["state"]["predecessorIdentifier"]
    del axiell_work["state"]["modifiedTime"]
    del axiell_work["version"]
    del calm_work["version"]
    if len(calm_work["state"]["relations"]["children"]) == 0:
        del calm_work["state"]["relations"]["children"]
    axiell_work["data"]["description"] = axiell_work["data"]["description"].replace("<p>", "").replace("</p>", "")

    return calm_work, axiell_work


i = 0
for work_id, work in transformed_axiell_works.items():
    i += 1
    if i > 10:
        break

    if work.state.predecessor_identifier:
        calm_id = work.state.predecessor_identifier.value
        with open(f"{SOURCE_WORKS_ROOT}/calm-record-id/{calm_id}.json") as f:
            calm_work = json.load(f)
            axiell_work = work.model_dump(exclude_none=True)

            calm_work, axiell_work = standardise_works(calm_work, axiell_work)

            print(work_id, DeepDiff(calm_work, axiell_work))
    # print(work.model_dump(exclude_none=True))


collect:110163064 {'dictionary_item_added': ["root['data']['description']", "root['data']['production'][0]['function']", "root['data']['production'][0]['dates'][0]['type']", "root['state']['mergeCandidates'][0]['id']['type']"], 'values_changed': {"root['data']['title']": {'new_value': 'Off-prints and reprints of articles by scientists', 'old_value': 'Miscellaneous correspondence'}, "root['data']['otherIdentifiers'][0]['value']": {'new_value': 'PPNHE/E/2/11', 'old_value': 'SANBT/R/R/16/21/2'}, "root['data']['otherIdentifiers'][1]['value']": {'new_value': 'PP/NHE/E/2/11', 'old_value': 'SA/NBT/R.21/2'}, "root['data']['production'][0]['label']": {'new_value': '1931-1933', 'old_value': 'Sep 1938-May 1942'}, "root['data']['production'][0]['dates'][0]['label']": {'new_value': '1931-1933', 'old_value': 'Sep 1938-May 1942'}, "root['data']['production'][0]['dates'][0]['range']['from']": {'new_value': '1931-01-01T00:00:00Z', 'old_value': '1938-09-01T00:00:00Z'}, "root['data']['production'][0]['da

AttributeError: 'dict' object has no attribute 'state'

In [22]:
print(axiell_to_calm)
print(ET.tostring(axiell_works["collect:100000183"], pretty_print=True, encoding="unicode"))

None
<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" priref="100000183" created="2026-05-15T14:55:40Z" modification="2026-05-15T14:55:40Z" selected="false" deleted="false">
  <access_category.notes>Restricted until 01/01/2025 for Data Protection reasons</access_category.notes>
  <accession_date>2008-08-13</accession_date>
  <acquisition.method>gift</acquisition.method>
  <acquisition.method.lref>3</acquisition.method.lref>
  <dataset_name>
    <value lang="neutral">accessions</value>
    <value lang="0">Accessions</value>
    <value lang="1">Inschrijvingen</value>
    <value lang="3">Zugänge</value>
  </dataset_name>
  <edit.date>2026-05-15</edit.date>
  <edit.name>Axiell</edit.name>
  <edit.notes>Migration Calm to 6.0</edit.notes>
  <edit.time>16:55:40</edit.time>
  <extent>1 volume</extent>
  <Content_description>
    <content.description>Preprinted Register under the Dangerous Drugs Acts, 1920-1925, and consolidated regulati

KeyError: '81ab1612-9f2b-432a-bc13-16fa53d55b09'